## Phase 3: Heuristic Hybrid Re-ranking

Two-stage retrieval: FAISS surfaces a broad semantic candidate pool (top 50), then a re-ranker
narrows it down using signals FAISS distance alone can't capture - community rating and
popularity. The re-ranker below uses fixed, hand-picked weights. Phase 5 replaces these
heuristic weights with weights learned from data.

This is a heuristic re-ranking baseline, not a learned Learning-to-Rank model.


In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sentence_transformers import SentenceTransformer
import faiss
import math
import json
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
import string
import pickle
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from collections import Counter
from difflib import get_close_matches

/home/tommaso/projects/Board-Game-Data-Analytics/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Persisting embeddings, the FAISS index, and BM25 tokens

Re-encoding ~20k descriptions, rebuilding the HNSW index, and re-tokenizing for BM25 each
take real time and produce the exact same result every time they're re-run on the same data,
so all three are cached to disk on the first run and reloaded on every subsequent one - each
in its own dedicated, single-purpose file:

- **`data/embeddings.npy`** - the raw embedding matrix (`np.save`/`np.load`), row-aligned
  with `df` by position.
- **`data/faiss_index.bin`** - the FAISS `IndexHNSWFlat`, saved with `faiss.write_index()` /
  reloaded with `faiss.read_index()`.
- **`data/tokens.pkl`** - the tokenized descriptions used to build the BM25 index (see the
  BM25 section below), saved as a pickled list of token lists, row-aligned with `df`.

`df` itself is **not** cached: merging the source CSVs takes a couple of seconds, so it's
always rebuilt fresh from `cleaned_dataset.csv` + `DM1_game_dataset.csv`.

Each cached artifact is also checked against `len(df)` on load (row count must match)
before being trusted.

If you change the embedding model, the source CSVs, or the preprocessing, delete
`embeddings.npy`, `faiss_index.bin`, and/or `tokens.pkl`so they get rebuilt.

### Re-ranking function

Weights (`0.5` semantic / `0.3` rating / `0.2` popularity) are a manual heuristic, not learned -
a reasonable starting point given no labeled training data exists yet at this point in the
notebook. It's the baseline the Phase 5 Ridge model is later compared against.

In [2]:
DATA_DIR = '../data'
EMBEDDINGS_PATH = os.path.join(DATA_DIR, 'embeddings.npy')
FAISS_INDEX_PATH = os.path.join(DATA_DIR, 'faiss_index.bin')

model = SentenceTransformer('all-MiniLM-L6-v2')

# df is always rebuilt fresh.
df_clean = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_dataset.csv'))
df_raw = pd.read_csv(os.path.join(DATA_DIR, 'DM1_game_dataset.csv'))
df_text = df_raw[['BGGId', 'Description']]
df = pd.merge(df_clean, df_text, on='BGGId', how='inner')
df = df.dropna(subset=['Description']).reset_index(drop=True)
print(f"Dataset ready! Total games: {len(df)}")

# Embeddings: load from cache if present and row-count-valid, else compute 
embeddings = None
if os.path.exists(EMBEDDINGS_PATH):
    cached_embeddings = np.load(EMBEDDINGS_PATH)
    if cached_embeddings.shape[0] == len(df):
        embeddings = cached_embeddings.astype('float32')
        print(f"Embeddings loaded from cache! Shape: {embeddings.shape}")
    else:
        print(
            f"Cache found at '{EMBEDDINGS_PATH}' but row count doesn't match df "
            f"({cached_embeddings.shape[0]} vs {len(df)}) - treating as stale, will recompute."
        )

if embeddings is None:
    print("Encoding descriptions for the LTR engine...")
    embeddings = model.encode(df['Description'].tolist(), show_progress_bar=True)
    embeddings = np.array(embeddings).astype('float32')
    np.save(EMBEDDINGS_PATH, embeddings)
    print(f"Embeddings computed and cached to '{EMBEDDINGS_PATH}'. Shape: {embeddings.shape}")

# FAISS index (HNSW): load from cache if present and vector-count-valid, else build
dimension = embeddings.shape[1]
index = None
if os.path.exists(FAISS_INDEX_PATH):
    cached_index = faiss.read_index(FAISS_INDEX_PATH)
    if cached_index.ntotal == len(df):
        index = cached_index
        print(f"FAISS index loaded from cache! Total vectors: {index.ntotal}")
    else:
        print(
            f"Cache found at '{FAISS_INDEX_PATH}' but vector count doesn't match df "
            f"({cached_index.ntotal} vs {len(df)}) - treating as stale, will rebuild."
        )

if index is None:
    index = faiss.IndexHNSWFlat(dimension, 32)
    print("Building the HNSW graph...")
    index.add(embeddings)
    faiss.write_index(index, FAISS_INDEX_PATH)
    print(f"FAISS index built and cached to '{FAISS_INDEX_PATH}'. Total vectors: {index.ntotal}")

print("Setup complete! You can now run the LTR search function.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14504.88it/s]


Dataset ready! Total games: 19728
Embeddings loaded from cache! Shape: (19728, 384)
FAISS index loaded from cache! Total vectors: 19728
Setup complete! You can now run the LTR search function.


In [3]:
def search_and_rerank_ltr(query, top_candidates=50, final_top_k=5):
    """
    Two-Stage Retrieval: Retrieves candidates via FAISS, then re-ranks them using LTR features.
    """
    # STAGE 1: Fast Candidate Retrieval (FAISS)
    query_vector = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vector, top_candidates)
    
    # Extract the candidate dataframe
    candidates = df.iloc[indices[0]].copy()
    candidates['faiss_distance'] = distances[0]
    
    # STAGE 2: Feature Extraction & Normalization
    scaler = MinMaxScaler()
    
    # 1. Normalize FAISS Distance (Invert it so Higher = Better)
    # Subtract from 1 so a distance of 0 becomes a score of 1.0
    candidates['semantic_score'] = 1 - scaler.fit_transform(candidates[['faiss_distance']])
    
    # 2. Normalize Numerical Metadata (from original Data Mining dataset)
    candidates['norm_rating'] = scaler.fit_transform(candidates[['Rating']])
    candidates['norm_popularity'] = scaler.fit_transform(candidates[['Avg_popularity_log']])
    
    # STAGE 3: Pointwise LTR Inference
    # In a fully supervised system, LambdaMART/XGBoost learns these weights.
    # Here, the inference of a trained pointwise ranker is simulated.
    W_SEMANTIC = 0.50    # Text relevance is the most important
    W_RATING = 0.30      # Game quality
    W_POPULARITY = 0.20  # How well-known it is
    
    # Calculate the final ranking score
    candidates['final_ltr_score'] = (
        (candidates['semantic_score'] * W_SEMANTIC) +
        (candidates['norm_rating'] * W_RATING) +
        (candidates['norm_popularity'] * W_POPULARITY)
    )
    
    # Sort by the new LTR score and return the top K
    final_results = candidates.sort_values(by='final_ltr_score', ascending=False).head(final_top_k)
    
    return final_results[['Name', 'final_ltr_score', 'semantic_score', 'norm_rating', 'Rating', 'Description']]

### Test the LTR engine
Same query used throughout notebooks 6-8, so results are directly comparable across BM25,
pure FAISS, and this re-ranked version.

In [4]:
user_query = "A cooperative space game involving betrayal"
print(f"LTR Re-ranked Results for: '{user_query}'\n")

ltr_results = search_and_rerank_ltr(user_query, top_candidates=50, final_top_k=5)

for i, row in ltr_results.iterrows():
    print(f"{row['Name']}")
    print(f"[LTR Score: {row['final_ltr_score']:.2f} | Semantic: {row['semantic_score']:.2f} | Rating: {row['Rating']:.1f}]")
    print(f"Snippet: {row['Description'][:150]}...\n")

LTR Re-ranked Results for: 'A cooperative space game involving betrayal'

The Crew: The Quest for Planet Nine
[LTR Score: 1.00 | Semantic: 1.00 | Rating: 3.0]
Snippet: cooperative tricktaking game crew quest planet player set astronaut uncertain space adventure rumor unknown planet eventful journey space extend   exc...

Cosmic Encounter
[LTR Score: 0.81 | Semantic: 0.62 | Rating: 3.0]
Snippet: build galactic empirein depth space alien race cosmo vie control universe alliance form shift moment moment cataclysmic battle send starship scream wa...

Star Clicker
[LTR Score: 0.64 | Semantic: 0.62 | Rating: 3.0]
Snippet: star clicker new cooperative game christophe raimbault   author colt express   player kid try save planet evil alien parent mission far far awayldquoe...

The Expanse Board Game
[LTR Score: 0.62 | Semantic: 0.38 | Rating: 3.0]
Snippet: expanse board game base syfy television series focus politic conquest intrigue similar board game twilight struggle short playing time cardd

## Phase 4: System Evaluation (Effectiveness Measures)

Three hand-labeled queries, used as a first sanity check on the metric implementations before
scaling up to an LLM-as-a-Judge over 100 queries in Phase 4.1 - small enough to eyeball by hand,
big enough to catch an obviously broken MRR/NDCG calculation.

Relevance scale: 3 = perfect match, 2 = highly relevant, 1 = partially relevant, 0 = irrelevant.


In [5]:
# Ground Truth dictionary (Qrels)
qrels = {
    "A cooperative space game involving betrayal": {
        "Battlestar Galactica: The Board Game": 3,
        "Nemesis": 3,
        "Unfathomable": 2, # Betrayal, but not space (it's ocean/Cthulhu)
        "The Crew: The Quest for Planet Nine": 1, # Space and coop, but no betrayal
        "Catan": 0
    },
    "Fast paced card game for two players": {
        "7 Wonders Duel": 3,
        "Jaipur": 3,
        "Lost Cities": 3,
        "Dominion": 2, # Card game, supports 2 players, but maybe not purely "fast paced 2p only"
        "Twilight Imperium": 0 # Exact opposite
    },
    "Heavy economic eurogame with route building": {
        "Brass: Birmingham": 3,
        "Brass: Lancashire": 3,
        "Great Western Trail": 3,
        "Ticket to Ride": 1, # Route building, but very light (not heavy economic)
        "Monopoly": 0
    }
}

test_queries = list(qrels.keys())
print(f"Loaded {len(test_queries)} test queries.")

Loaded 3 test queries.


## Metric Calculation Functions

MRR and NDCG implemented directly rather than pulled from a library, mainly for control over
tie-breaking and to keep the exact rank-discounting logic visible for the report. The two are
complementary: MRR only cares about the position of the *first* relevant hit, NDCG grades the
whole top-k by relevance level - a system can win on one and lose on the other (see the Phase 5
discussion below).

In [6]:
def calculate_mrr(retrieved_games, query_qrels):
    """
    Calculates the Reciprocal Rank for a single query.
    Considers a document 'relevant' if its qrel score is > 0.
    """
    for rank, game_name in enumerate(retrieved_games, start=1):
        # Check if the game is in our ground truth and has a score > 0
        if query_qrels.get(game_name, 0) > 0:
            return 1.0 / rank
    return 0.0

def calculate_ndcg(retrieved_games, query_qrels, k=5):
    """
    Calculates NDCG@K for a single query.
    """
    # 1. Calculate DCG
    dcg = 0.0
    for i in range(min(k, len(retrieved_games))):
        game_name = retrieved_games[i]
        rel_score = query_qrels.get(game_name, 0)
        # Using the standard DCG formula
        dcg += rel_score / math.log2((i + 1) + 1)
        
    # 2. Calculate IDCG (Ideal DCG)
    # Sort the ground truth scores in descending order
    ideal_scores = sorted(list(query_qrels.values()), reverse=True)
    idcg = 0.0
    for i in range(min(k, len(ideal_scores))):
        idcg += ideal_scores[i] / math.log2((i + 1) + 1)
        
    # Prevent division by zero
    if idcg == 0.0:
        return 0.0
        
    # 3. NDCG
    return dcg / idcg

## Model Comparison: BM25 vs Heuristic Hybrid Re-ranking

Checks whether the FAISS + heuristic re-ranking pipeline actually beats plain lexical search
(BM25) on this small labeled set, before scaling up to the expanded evaluation set.


In [7]:
TOKENS_PATH = os.path.join(DATA_DIR, 'tokens.pkl')

# 1. Define the preprocessing function
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    return [word for word in tokens if word not in stop_words]

# 2. Tokenize the descriptions - reuse the cached tokens if present and valid,
#    otherwise compute once and persist to their own dedicated file.
tokens_list = None
if os.path.exists(TOKENS_PATH):
    with open(TOKENS_PATH, 'rb') as f:
        cached_tokens = pickle.load(f)
    if len(cached_tokens) == len(df):
        tokens_list = cached_tokens
        print("Reusing cached tokens from disk...")
    else:
        print(
            f"Cache found at '{TOKENS_PATH}' but row count doesn't match df "
            f"({len(cached_tokens)} vs {len(df)}) - treating as stale, will recompute."
        )

if tokens_list is None:
    print("Tokenizing descriptions for BM25 (takes a minute)...")
    tokens_list = df['Description'].apply(preprocess_text).tolist()
    with open(TOKENS_PATH, 'wb') as f:
        pickle.dump(tokens_list, f)
    print(f"Tokens computed and cached to '{TOKENS_PATH}'.")

df['tokens'] = tokens_list

# 3. Build the BM25 Index
print("Building BM25 Index...")
bm25 = BM25Okapi(df['tokens'].tolist())

# 4. Define the BM25 search function
def search_games_bm25(query, top_k=5):
    query_tokens = preprocess_text(query)
    doc_scores = bm25.get_scores(query_tokens)

    # We use a copy of the dataframe here to avoid modifying the original one during testing
    df_temp = df.copy()
    df_temp['bm25_score'] = doc_scores
    results = df_temp.sort_values(by='bm25_score', ascending=False).head(top_k)
    return results

print("BM25 is now active in memory! You can run the evaluation cell.")

Reusing cached tokens from disk...
Building BM25 Index...
BM25 is now active in memory! You can run the evaluation cell.


In [8]:
bm25_mrr_total = 0
bm25_ndcg_total = 0
ltr_mrr_total = 0
ltr_ndcg_total = 0

print("Starting Evaluation...\n")

for query in test_queries:
    print(f"Testing Query: '{query}'")
    query_qrels = qrels[query]
    
    # -- 1. Test BM25 --
    # Assuming search_games_bm25 is defined and returns a dataframe
    bm25_results = search_games_bm25(query, top_k=5)['Name'].tolist()
    bm25_mrr = calculate_mrr(bm25_results, query_qrels)
    bm25_ndcg = calculate_ndcg(bm25_results, query_qrels, k=5)
    
    # -- 2. Test LTR --
    ltr_results = search_and_rerank_ltr(query, top_candidates=50, final_top_k=5)['Name'].tolist()
    ltr_mrr = calculate_mrr(ltr_results, query_qrels)
    ltr_ndcg = calculate_ndcg(ltr_results, query_qrels, k=5)
    
    # Accumulate totals
    bm25_mrr_total += bm25_mrr
    bm25_ndcg_total += bm25_ndcg
    ltr_mrr_total += ltr_mrr
    ltr_ndcg_total += ltr_ndcg
    
    print(f"  BM25 -> MRR: {bm25_mrr:.2f} | NDCG@5: {bm25_ndcg:.2f}")
    print(f"  LTR  -> MRR: {ltr_mrr:.2f} | NDCG@5: {ltr_ndcg:.2f}\n")

# Calculate Averages
num_queries = len(test_queries)

print("FINAL AVERAGE SCORES")

print(f"BM25 Baseline:")
print(f"  Mean MRR:    {bm25_mrr_total / num_queries:.3f}")
print(f"  Mean NDCG@5: {bm25_ndcg_total / num_queries:.3f}")
print(f"\nLTR Pipeline:")
print(f"  Mean MRR:    {ltr_mrr_total / num_queries:.3f}")
print(f"  Mean NDCG@5: {ltr_ndcg_total / num_queries:.3f}")

Starting Evaluation...

Testing Query: 'A cooperative space game involving betrayal'
  BM25 -> MRR: 0.00 | NDCG@5: 0.00
  LTR  -> MRR: 1.00 | NDCG@5: 0.16

Testing Query: 'Fast paced card game for two players'
  BM25 -> MRR: 0.00 | NDCG@5: 0.00
  LTR  -> MRR: 0.00 | NDCG@5: 0.00

Testing Query: 'Heavy economic eurogame with route building'
  BM25 -> MRR: 0.00 | NDCG@5: 0.00
  LTR  -> MRR: 0.00 | NDCG@5: 0.00

FINAL AVERAGE SCORES
BM25 Baseline:
  Mean MRR:    0.000
  Mean NDCG@5: 0.000

LTR Pipeline:
  Mean MRR:    0.333
  Mean NDCG@5: 0.053


## Phase 4.1: Automated Qrel Generation (LLM-as-a-Judge)

Hand-labeling qrels doesn't scale past a handful of queries, so the expanded evaluation set
(100 queries) is generated with an LLM judge (`gemini-3.5-flash-lite` when available).
Candidates are pooled from the top 20 of BM25 and the top 20 of pure FAISS retrieval per query.
The two lists are merged, so the judge evaluates only candidates surfaced by at least one of
the two base retrieval systems.

**Ranking, not scoring.** The model ranks the whole pool for a query rather than assigning a
raw 0-3 score to each game independently; the score is then derived deterministically from rank
position (top 2 -> 3, next 6 -> 2, next 8 -> 1, rest -> 0). Asking for a raw score directly
lets the model's idea of relevance drift between queries; asking it to rank the complete pool
makes the subsequent bucket mapping deterministic and consistent across queries.

These are LLM-generated graded relevance judgments, not human ground truth. They are also
pool-based: games outside the BM25/FAISS candidate pool are not judged.

Every query is generated with the same method and model. A checkpoint built partially with a
different method/model is discarded rather than mixed in.

The API call itself - rate-limiting, retries, JSON parsing - lives in
`scripts/generate_qrels_expanded.py` and runs as a separate batch job. It produces
`expanded_qrels.json`; it does not belong inline in a notebook meant to be re-run interactively.


In [9]:
# Load the pre-generated Qrels from data/evaluation/
with open('../data/evaluation/expanded_qrels.json', 'r') as f:
    saved = json.load(f)
full_qrels = saved.get('qrels', saved)

print(f"Successfully loaded {len(full_qrels)} annotated queries!")

Successfully loaded 100 annotated queries!


### Verifying the qrels

Three checks before using these LLM-generated judgments: judged names match `df['Name']`
exactly, every query follows the deterministic 2/6/8 relevance buckets, and a manual
spot-check of a few queries can catch cases where the judge appears too generous or otherwise
unexpected.

The expected distribution for a pool of N judged games is:
- 2 games scored 3
- 6 games scored 2
- 8 games scored 1
- N - 16 games scored 0


In [10]:
known_names = set(df['Name'])
N_SCORE_3 = 2
N_SCORE_2 = 6
N_SCORE_1 = 8

with open('../data/evaluation/expanded_qrels.json', 'r') as f:
    saved = json.load(f)
qrels = saved['qrels']
print(f"Method: {saved['_metadata'].get('method')} | Model: {saved['_metadata'].get('model')}")

print(f"Total queries: {len(qrels)}")  # expected: 100
total_judgments = sum(len(v) for v in qrels.values())
print(f"Total judgments: {total_judgments}\n")

# 1. Exact name match against df['Name'], across all 100 queries
all_ok = True
for query, scores in qrels.items():
    mismatches = [name for name in scores if name not in known_names]
    if mismatches:
        all_ok = False
        print(f"Query: '{query}'")
        for name in mismatches:
            close = get_close_matches(name, known_names, n=2, cutoff=0.6)
            print(f"  [MISMATCH] '{name}' (score {scores[name]}) -> possible: {close}")
        print()
if all_ok:
    print("All judged names match df['Name'] exactly across all 100 queries.\n")

# 2. Verify the deterministic 2/6/8 bucket mapping.
# For a pool of N judged games, the expected counts are:
# 3 -> 2 games, 2 -> 6 games, 1 -> 8 games, 0 -> N-16 games.
bucket_ok = True
for query, scores in qrels.items():
    dist = Counter(scores.values())
    expected = {
        3: min(N_SCORE_3, len(scores)),
        2: min(N_SCORE_2, max(0, len(scores) - N_SCORE_3)),
        1: min(N_SCORE_1, max(0, len(scores) - N_SCORE_3 - N_SCORE_2)),
    }
    expected[0] = max(0, len(scores) - N_SCORE_3 - N_SCORE_2 - N_SCORE_1)

    actual = {score: dist.get(score, 0) for score in [3, 2, 1, 0]}
    if actual != expected:
        bucket_ok = False
        print(f"[BUCKET CHECK] '{query}'")
        print(f"  Expected: {expected}")
        print(f"  Actual:   {actual}")

if bucket_ok:
    print("All queries follow the expected 2/6/8 bucket mapping.")


Method: ranking_bucket_expanded_v3 | Model: gemini-3.5-flash-lite
Total queries: 100
Total judgments: 3622

All judged names match df['Name'] exactly across all 100 queries.

All queries follow the expected 2/6/8 bucket mapping.


Full listing below, sorted by relevance, for a manual spot-check of the flagged queries (or any
others).

In [11]:
for query, scores in qrels.items():
    print(f"\n{query}")
    for game, score in sorted(scores.items(), key=lambda x: -x[1]):
        print(f"  {score}  {game}")


Cyberpunk game with hacking mechanics
  3  Android: Netrunner
  3  Shadowrun Crossfire: Prime Runner Edition
  2  Shadowrun: Crossfire
  2  Shadowrun: Sprawl Ops
  2  Shadowrun: The Trading Card Game
  2  Cypher
  2  League of Hackers
  2  Super Hack Override
  1  Hacker
  1  Hacker: Deluxe Edition
  1  Intelle
  1  Black Hat
  1  HeroCard Cyberspace
  1  Human Interface: Be a Better Human
  1  Zaibatsu
  1  Cyberpunk: The Collectible Card Game
  0  .hack//ENEMY
  0  Bitcoin Hackers
  0  Sabotage
  0  Rogue Agent
  0  Martian Rails
  0  Aetherium
  0  Black Ops: Tactical Espionage Wargaming
  0  Gamedec
  0  RoboDerby: Express
  0  Evil Intent
  0  Streets Of Steel: Kickin' Asphalt
  0  Alien Puppies
  0  Nodwick: The Card Game
  0  ESSEN
  0  Set & Match
  0  L-Game
  0  Fabled Fruit
  0  Mees, kes teadis ussisõnu
  0  Green Box of Games
  0  Stonehenge: An Anthology Board Game
  0  Electronic Stratego
  0  Sedition Wars: Battle for Alabaster

Lovecraftian horror game set in the 1920

## Phase 4.2: Large-Scale Evaluation (BM25 vs. LTR)

Same comparison as Phase 4, now over 100 LLM-judged queries instead of 3 hand-labeled ones -
enough to say something about average behavior rather than reading into 3 data points.

In [12]:
# 1. Load the automated Qrels
with open('../data/evaluation/expanded_qrels.json', 'r') as f:
    saved = json.load(f)
full_qrels = saved.get('qrels', saved)

# 2. Initialize the metric counters
bm25_mrr_total, bm25_ndcg_total = 0, 0
hybrid_mrr_total, hybrid_ndcg_total = 0, 0
num_queries = len(full_qrels)

print(f"Evaluating BM25 vs. Heuristic Hybrid Re-ranking on {num_queries} queries...\n")

# 3. Evaluation Loop
for query, query_qrels in full_qrels.items():

    # --- 1. Test BM25 ---
    bm25_results = search_games_bm25(query, top_k=5)['Name'].tolist()
    bm25_mrr = calculate_mrr(bm25_results, query_qrels)
    bm25_ndcg = calculate_ndcg(bm25_results, query_qrels, k=5)

    # --- 2. Test Phase 3 heuristic hybrid re-ranking ---
    hybrid_results = search_and_rerank_ltr(query, top_candidates=20, final_top_k=5)['Name'].tolist()
    hybrid_mrr = calculate_mrr(hybrid_results, query_qrels)
    hybrid_ndcg = calculate_ndcg(hybrid_results, query_qrels, k=5)

    # Accumulate the results
    bm25_mrr_total += bm25_mrr
    bm25_ndcg_total += bm25_ndcg
    hybrid_mrr_total += hybrid_mrr
    hybrid_ndcg_total += hybrid_ndcg

# 4. Print the final results
print(f"FINAL AVERAGE RESULTS ({len(full_qrels)} QUERIES)")

print("BM25 Baseline (Lexical Search):")
print(f"  Mean MRR:    {bm25_mrr_total / num_queries:.3f}")
print(f"  Mean NDCG@5: {bm25_ndcg_total / num_queries:.3f}")

print("\nHeuristic Hybrid Re-ranking:")
print(f"  Mean MRR:    {hybrid_mrr_total / num_queries:.3f}")
print(f"  Mean NDCG@5: {hybrid_ndcg_total / num_queries:.3f}")


Evaluating BM25 vs. Heuristic Hybrid Re-ranking on 100 queries...

FINAL AVERAGE RESULTS (100 QUERIES)
BM25 Baseline (Lexical Search):
  Mean MRR:    0.889
  Mean NDCG@5: 0.625

Heuristic Hybrid Re-ranking:
  Mean MRR:    0.756
  Mean NDCG@5: 0.410


## Phase 5: Machine-Learned Ranking (Hybrid Search with Linear Regression)

Learns the re-ranking weights instead of hand-picking them, using the LLM-judged queries as
training signal (split by query so no game from a test query leaks
into training). Features: `bm25_score`, `semantic_score`, `norm_rating`, `norm_popularity` -
the same signal types as the Phase 3 heuristic, plus BM25, so this pool blends lexical and
semantic retrieval rather than starting from FAISS candidates alone.

Ridge (`alpha=1.0`) rather than plain linear regression: with training queries and 4
features, an unregularized fit risks weights that swing on noise; L2 regularization keeps them
stable. `linear_ranker.coef_` prints the actual learned weights for direct comparison against
the fixed `0.5/0.3/0.2` used in Phase 3.

In [13]:
print("1. Building the LTR Training Dataset...")

def extract_query_features(query, query_qrels):
    # IMPORTANT: use the qrels themselves as the authoritative candidate pool.
    # The qrel-generation script judged exactly the union of BM25 top-20 and
    # pure-FAISS top-20 for each query. Re-running an approximate HNSW search
    # here can return a slightly different top-20 (especially with a different
    # cached/rebuilt index), which would introduce candidates with no judgment.
    # Using the qrel names directly guarantees that every LTR example is judged.
    judged_names = set(query_qrels.keys())
    pool_df = df[df['Name'].isin(judged_names)].copy()

    missing_from_df = judged_names - set(pool_df['Name'])
    if missing_from_df:
        raise ValueError(
            f"{len(missing_from_df)} qrel game names are missing from df for query "
            f"'{query}': {sorted(list(missing_from_df))[:10]}"
        )

    # Recalculate raw BM25 scores for the exact judged pool.
    query_tokens = preprocess_text(query)
    all_bm25_scores = bm25.get_scores(query_tokens)
    pool_df['bm25_raw'] = all_bm25_scores[pool_df.index]

    # Compute FAISS-space distances for the exact judged documents.
    # This does not perform retrieval; it only extracts the semantic feature.
    query_vector = model.encode([query]).astype('float32')
    pool_indices = pool_df.index.to_numpy()
    pool_embeddings = embeddings[pool_indices]
    faiss_distances = np.linalg.norm(pool_embeddings - query_vector, axis=1)
    pool_df['faiss_raw'] = faiss_distances

    # --- FEATURE NORMALIZATION ---
    scaler = MinMaxScaler()

    # BM25: Higher is better
    pool_df['bm25_score'] = scaler.fit_transform(pool_df[['bm25_raw']])

    # FAISS: Lower is better (distance), so we invert it (1 - normalized distance)
    pool_df['semantic_score'] = 1 - scaler.fit_transform(pool_df[['faiss_raw']])

    # Global metadata
    pool_df['norm_rating'] = scaler.fit_transform(pool_df[['Rating']])
    pool_df['norm_popularity'] = scaler.fit_transform(pool_df[['Avg_popularity_log']])

    # --- TARGET (LLM-GENERATED GRADED RELEVANCE JUDGMENT) ---
    # Every candidate here is explicitly present in the qrels.
    pool_df['relevance_label'] = pool_df['Name'].map(query_qrels)

    if pool_df['relevance_label'].isna().any():
        missing_labels = pool_df.loc[
            pool_df['relevance_label'].isna(), 'Name'
        ].tolist()
        raise ValueError(
            f"Missing qrel labels for {len(missing_labels)} candidates in query "
            f"'{query}': {missing_labels[:10]}"
        )

    pool_df['query'] = query

    return pool_df[
        ['query', 'Name', 'bm25_score', 'semantic_score',
         'norm_rating', 'norm_popularity', 'relevance_label']
    ]

# Assemble the dataset for all queries
all_data = []
for query, qrels in full_qrels.items():
    query_df = extract_query_features(query, qrels)
    all_data.append(query_df)

ltr_dataset = pd.concat(all_data, ignore_index=True)
print(f"Dataset created: {len(ltr_dataset)} (Query, Game) pairs extracted.")

# 2. Train/Test Split based on QUERIES (Not individual rows!)
queries = list(full_qrels.keys())
train_queries, test_queries = train_test_split(
    queries, test_size=0.15, random_state=42
)

train_data = ltr_dataset[ltr_dataset['query'].isin(train_queries)]
test_data = ltr_dataset[ltr_dataset['query'].isin(test_queries)]

features = ['bm25_score', 'semantic_score', 'norm_rating', 'norm_popularity']
X_train = train_data[features]
y_train = train_data['relevance_label']

# 3. Train the Linear Model (Ridge)
print("\n2. Training Ridge Regression (Pointwise LTR)...")
linear_ranker = Ridge(alpha=1.0)
linear_ranker.fit(X_train, y_train)

print("\nWEIGHTS LEARNED BY THE MODEL")
for feature, weight in zip(features, linear_ranker.coef_):
    print(f"  {feature}: {weight:.3f}")

# 4. Evaluation on the held-out query set
num_test = len(test_queries)
print(f"\n3. Evaluation on the Test Set ({num_test} Held-Out Queries)...")
mlr_mrr_total, mlr_ndcg_total = 0, 0
bm25_test_mrr, bm25_test_ndcg = 0, 0

for query in test_queries:
    query_qrels = full_qrels[query]
    query_test_data = test_data[test_data['query'] == query].copy()

    # BM25 Baseline
    bm25_results = query_test_data.sort_values(
        'bm25_score', ascending=False
    ).head(5)['Name'].tolist()
    bm25_test_mrr += calculate_mrr(bm25_results, query_qrels)
    bm25_test_ndcg += calculate_ndcg(bm25_results, query_qrels, k=5)

    # Learned Ridge pointwise LTR
    query_test_data['pred_score'] = linear_ranker.predict(
        query_test_data[features]
    )
    mlr_results = query_test_data.sort_values(
        'pred_score', ascending=False
    ).head(5)['Name'].tolist()
    mlr_mrr_total += calculate_mrr(mlr_results, query_qrels)
    mlr_ndcg_total += calculate_ndcg(mlr_results, query_qrels, k=5)

print(f"\nTEST SET RESULTS ({num_test} QUERIES)")
print("BM25 Baseline:")
print(f"  Mean MRR:    {bm25_test_mrr / num_test:.3f}")
print(f"  Mean NDCG@5: {bm25_test_ndcg / num_test:.3f}")
print("\nRidge Pointwise LTR (Hybrid Search):")
print(f"  Mean MRR:    {mlr_mrr_total / num_test:.3f}")
print(f"  Mean NDCG@5: {mlr_ndcg_total / num_test:.3f}")


1. Building the LTR Training Dataset...
Dataset created: 3747 (Query, Game) pairs extracted.

2. Training Ridge Regression (Pointwise LTR)...

WEIGHTS LEARNED BY THE MODEL
  bm25_score: 0.973
  semantic_score: 0.447
  norm_rating: 0.258
  norm_popularity: 0.470

3. Evaluation on the Test Set (15 Held-Out Queries)...

TEST SET RESULTS (15 QUERIES)
BM25 Baseline:
  Mean MRR:    0.900
  Mean NDCG@5: 0.644

Ridge Pointwise LTR (Hybrid Search):
  Mean MRR:    0.956
  Mean NDCG@5: 0.665


The learned hybrid ranker achieves higher MRR and NDCG@5 than the BM25
baseline on the held-out queries (MRR 0.956 vs. 0.900; NDCG@5 0.665 vs. 0.644). This suggests
that combining lexical matching with semantic and metadata-based features can improve ranking
quality on this evaluation set.

The improvement is clear, however, the test set contains only 15 held-out queries. In
addition, the relevance judgments are LLM-generated, pool-based, and derived from a relative
ranking rather than from exhaustive human annotation. The results should therefore be
interpreted as strong evidence that the learned hybrid approach is promising, rather than as
statistically conclusive evidence across all possible user intents.
